# Conformal Prediction for Crop AP Ratio

This notebook adds the uncertainty-quantification part of the proposed architecture without changing the main model-comparison notebook.

It follows the same dataset target and split philosophy used in `crop_ap_ratio_prediction.ipynb`:

- target: `AP Ratio`
- point model: CatBoost-style tabular regression on the existing engineered features
- calibration: validation fold residuals
- uncertainty methods: naive Gaussian intervals, split conformal intervals, and optional group-conditional conformal intervals

The key idea is simple: after fitting the model on the training fold, we use the validation fold only to estimate how large prediction errors tend to be. That residual scale is then converted into prediction intervals on the test fold.


## 1. Imports


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
from statistics import NormalDist

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor

try:
    from mapie.regression import MapieRegressor
    MAPIE_AVAILABLE = True
except Exception:
    MapieRegressor = None
    MAPIE_AVAILABLE = False

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", None)

print("Imports OK")
print(f"MAPIE available: {MAPIE_AVAILABLE}")


## 2. Load Engineered Data

This companion notebook starts from `dataset/crop_engineered.csv`, which is produced by the main prediction notebook. That keeps this notebook focused on conformal prediction rather than repeating all feature-engineering code.

`AP Ratio` remains the target. `District`, `Season`, and `season_order` are kept for OOD splitting. Raw crop labels are loaded from `preprocessed_data.csv` only for subgroup coverage diagnostics; they are not used as model inputs.


In [ ]:
DATA_PATH = "../dataset/crop_engineered.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        "dataset/crop_engineered.csv was not found. Run crop_ap_ratio_prediction.ipynb through the export cell first."
    )

df = pd.read_csv(DATA_PATH)

# crop_engineered.csv keeps District and Season, but not the raw Crop Name label.
# Load the cleaned table only for subgroup metadata; model features still come from crop_engineered.csv.
META_PATH = "../dataset/preprocessed_data.csv"
if os.path.exists(META_PATH):
    df_meta = pd.read_csv(META_PATH)
    if len(df_meta) != len(df):
        print("Metadata row count differs from engineered data; falling back to engineered metadata only.")
        df_meta = df.copy()
else:
    df_meta = df.copy()

# Add readable crop labels back into df for subgroup diagnostics.
# This column is metadata only and is excluded from model features below.
if "Crop Name" in df_meta.columns and len(df_meta) == len(df):
    df["Crop Name"] = df_meta["Crop Name"].values

print(f"Shape: {df.shape}")
print(df.columns.tolist())
df.head(3)


In [ ]:
TARGET = "AP Ratio"
META_COLS = ["District", "Season", "Crop Name"]

# Keep season_order as a model feature because the main notebook also uses it.
# District, Season, and Crop Name remain readable metadata for OOD splits and subgroup coverage.
NON_FEATURE_COLS = [TARGET] + [c for c in META_COLS if c in df.columns]
FEATURE_COLS = [c for c in df.columns if c not in NON_FEATURE_COLS]

X = df[FEATURE_COLS].copy()
y = df[TARGET].copy()

print(f"Feature matrix: {X.shape}")
print(f"Target summary: min={y.min():.4f}, mean={y.mean():.4f}, max={y.max():.4f}")
print("Feature columns:")
print(FEATURE_COLS)


## 3. Train / Calibration / Test Splits

Conformal prediction needs a calibration set that was not used to fit the model. We therefore reuse the same split pattern as the main notebook:

- random i.i.d. split: 70% train, 15% calibration, 15% test
- spatial OOD split: disjoint district groups
- season OOD split: Rabi for train, Kharif 1 for calibration, Kharif 2 for test

The validation fold is called `calibration` here because its uncertainty role is to calibrate intervals.


In [ ]:
# A. Random 70 / 15 / 15 split
X_tr_r, X_tmp, y_tr_r, y_tmp = train_test_split(X, y, test_size=0.30, random_state=SEED)
X_cal_r, X_te_r, y_cal_r, y_te_r = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=SEED)

# B. Spatial OOD split with disjoint districts
all_districts = df["District"].unique().copy()
np.random.shuffle(all_districts)

n_d = len(all_districts)
n_tr_d = int(0.70 * n_d)
n_cal_d = int(0.15 * n_d)

train_districts = set(all_districts[:n_tr_d])
cal_districts = set(all_districts[n_tr_d:n_tr_d + n_cal_d])
test_districts = set(all_districts[n_tr_d + n_cal_d:])

mask_tr_s = df["District"].isin(train_districts)
mask_cal_s = df["District"].isin(cal_districts)
mask_te_s = df["District"].isin(test_districts)

X_tr_s, y_tr_s = X[mask_tr_s.values], y[mask_tr_s.values]
X_cal_s, y_cal_s = X[mask_cal_s.values], y[mask_cal_s.values]
X_te_s, y_te_s = X[mask_te_s.values], y[mask_te_s.values]

# C. Season OOD split
mask_tr_season = df["season_order"] == 0
mask_cal_season = df["season_order"] == 1
mask_te_season = df["season_order"] == 2

X_tr_season, y_tr_season = X[mask_tr_season.values], y[mask_tr_season.values]
X_cal_season, y_cal_season = X[mask_cal_season.values], y[mask_cal_season.values]
X_te_season, y_te_season = X[mask_te_season.values], y[mask_te_season.values]

split_summary = pd.DataFrame([
    {"Split": "Random", "Train": len(X_tr_r), "Calibration": len(X_cal_r), "Test": len(X_te_r)},
    {"Split": "Spatial OOD", "Train": len(X_tr_s), "Calibration": len(X_cal_s), "Test": len(X_te_s)},
    {"Split": "Season OOD", "Train": len(X_tr_season), "Calibration": len(X_cal_season), "Test": len(X_te_season)},
])
split_summary


## 4. Scaling and CatBoost Training

The current main notebook scales all engineered numeric columns before fitting the model zoo, including CatBoost. To stay consistent with the existing code, this notebook uses the same train-only scaling pattern.

The calibration and test folds are transformed using the scaler fitted on the corresponding training fold only.


In [ ]:
def scale_split(X_tr, X_cal, X_te):
    scaler = StandardScaler()
    Xs_tr = pd.DataFrame(scaler.fit_transform(X_tr), columns=X_tr.columns, index=X_tr.index)
    Xs_cal = pd.DataFrame(scaler.transform(X_cal), columns=X_cal.columns, index=X_cal.index)
    Xs_te = pd.DataFrame(scaler.transform(X_te), columns=X_te.columns, index=X_te.index)
    return Xs_tr, Xs_cal, Xs_te, scaler


Xs_tr_r, Xs_cal_r, Xs_te_r, scaler_r = scale_split(X_tr_r, X_cal_r, X_te_r)
Xs_tr_s, Xs_cal_s, Xs_te_s, scaler_s = scale_split(X_tr_s, X_cal_s, X_te_s)
Xs_tr_season, Xs_cal_season, Xs_te_season, scaler_season = scale_split(X_tr_season, X_cal_season, X_te_season)

print("Scaling complete.")


In [ ]:
def compute_point_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }


def make_catboost():
    return CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=6,
        l2_leaf_reg=3.0,
        loss_function="RMSE",
        random_seed=SEED,
        allow_writing_files=False,
        verbose=False,
    )


def fit_catboost_for_split(split_name, Xs_tr, y_tr, Xs_cal, y_cal, Xs_te, y_te):
    model = make_catboost()
    model.fit(Xs_tr, y_tr)

    cal_pred = model.predict(Xs_cal)
    test_pred = model.predict(Xs_te)
    cal_metrics = compute_point_metrics(y_cal, cal_pred)
    test_metrics = compute_point_metrics(y_te, test_pred)

    print(f"{split_name}")
    print(f"  Calibration RMSE={cal_metrics['RMSE']:.4f}, R2={cal_metrics['R2']:.4f}")
    print(f"  Test        RMSE={test_metrics['RMSE']:.4f}, R2={test_metrics['R2']:.4f}")

    return {
        "model": model,
        "cal_pred": cal_pred,
        "test_pred": test_pred,
        "cal_metrics": cal_metrics,
        "test_metrics": test_metrics,
    }


catboost_results = {
    "Random": fit_catboost_for_split("Random", Xs_tr_r, y_tr_r, Xs_cal_r, y_cal_r, Xs_te_r, y_te_r),
    "Spatial OOD": fit_catboost_for_split("Spatial OOD", Xs_tr_s, y_tr_s, Xs_cal_s, y_cal_s, Xs_te_s, y_te_s),
    "Season OOD": fit_catboost_for_split("Season OOD", Xs_tr_season, y_tr_season, Xs_cal_season, y_cal_season, Xs_te_season, y_te_season),
}


## 5. Split Conformal Prediction

For a desired error level \(\alpha\), split conformal prediction computes absolute calibration residuals:

\[
r_i = |y_i - \hat{y}_i|
\]

and then chooses a corrected empirical quantile:

\[
q_\alpha = Q_{\lceil(n+1)(1-\alpha)\rceil/n}(r_1,\ldots,r_n).
\]

The prediction interval for a new point is:

\[
[\hat{y} - q_\alpha,\ \hat{y} + q_\alpha].
\]

Under exchangeability between calibration and test data, this gives finite-sample marginal coverage close to \(1-\alpha\). In OOD splits, exchangeability is intentionally stressed, so coverage is reported rather than assumed.


In [ ]:
def conformal_quantile(abs_residuals, alpha):
    residuals = np.asarray(abs_residuals, dtype=float)
    n = residuals.size
    if n == 0:
        raise ValueError("Calibration residuals are empty.")
    q_level = np.ceil((n + 1) * (1 - alpha)) / n
    q_level = min(q_level, 1.0)
    return float(np.quantile(residuals, q_level, method="higher"))


def interval_metrics(y_true, lower, upper):
    y_arr = np.asarray(y_true, dtype=float)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    covered = (y_arr >= lower) & (y_arr <= upper)
    return {
        "Coverage": float(covered.mean()),
        "Avg Width": float(np.mean(upper - lower)),
        "Median Width": float(np.median(upper - lower)),
    }


def gaussian_interval(cal_residuals, test_pred, alpha):
    sigma = float(np.std(cal_residuals, ddof=1))
    z = NormalDist().inv_cdf(1 - alpha / 2)
    half_width = z * sigma
    return test_pred - half_width, test_pred + half_width


def split_conformal_interval(cal_residuals, test_pred, alpha):
    q_alpha = conformal_quantile(np.abs(cal_residuals), alpha)
    return test_pred - q_alpha, test_pred + q_alpha, q_alpha


## 6. Global Interval Evaluation

This table compares the architecture's conformal interval against a simple Gaussian baseline. The main metrics are:

- **Coverage:** fraction of test targets inside the interval.
- **Average width:** mean interval width, measuring sharpness.
- **Median width:** robust interval width summary.


In [ ]:
def evaluate_intervals(split_name, y_cal, y_te, result, alpha_values=(0.10, 0.05)):
    cal_residuals = y_cal.values - result["cal_pred"]
    test_pred = result["test_pred"]

    rows = []
    intervals = {}
    for alpha in alpha_values:
        nominal = 1 - alpha

        lower_g, upper_g = gaussian_interval(cal_residuals, test_pred, alpha)
        rows.append({
            "Split": split_name,
            "Method": "Naive Gaussian",
            "Nominal": nominal,
            **interval_metrics(y_te, lower_g, upper_g),
        })
        intervals[(split_name, "Naive Gaussian", nominal)] = (lower_g, upper_g)

        lower_c, upper_c, q_alpha = split_conformal_interval(cal_residuals, test_pred, alpha)
        rows.append({
            "Split": split_name,
            "Method": "Split Conformal",
            "Nominal": nominal,
            "q_alpha": q_alpha,
            **interval_metrics(y_te, lower_c, upper_c),
        })
        intervals[(split_name, "Split Conformal", nominal)] = (lower_c, upper_c)

    return pd.DataFrame(rows), intervals


uq_frames = []
interval_lookup = {}

split_targets = {
    "Random": (y_cal_r, y_te_r),
    "Spatial OOD": (y_cal_s, y_te_s),
    "Season OOD": (y_cal_season, y_te_season),
}

for split_name, result in catboost_results.items():
    y_cal_split, y_te_split = split_targets[split_name]
    frame, intervals = evaluate_intervals(split_name, y_cal_split, y_te_split, result)
    uq_frames.append(frame)
    interval_lookup.update(intervals)

uq_results = pd.concat(uq_frames, ignore_index=True)
uq_results_display = uq_results.copy()
for col in ["Nominal", "Coverage", "Avg Width", "Median Width", "q_alpha"]:
    if col in uq_results_display.columns:
        uq_results_display[col] = uq_results_display[col].round(4)

uq_results_display


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.barplot(data=uq_results, x="Split", y="Coverage", hue="Method", ax=axes[0])
for nominal in sorted(uq_results["Nominal"].unique()):
    axes[0].axhline(nominal, color="gray", linestyle="--", linewidth=0.9)
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Empirical Coverage")
axes[0].tick_params(axis="x", rotation=15)

sns.barplot(data=uq_results, x="Split", y="Avg Width", hue="Method", ax=axes[1])
axes[1].set_title("Average Interval Width")
axes[1].tick_params(axis="x", rotation=15)
axes[1].legend_.remove()

plt.suptitle("CatBoost Prediction Interval Quality", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 7. Test-Set Outcome Demonstration

The table below shows what the model would actually return on held-out test samples. For each test record, we report:

- the true `AP Ratio`
- the CatBoost point prediction
- the 90% split-conformal lower and upper bounds
- interval width
- whether the true value falls inside the interval

This is the clearest demonstration of the final output: a yield prediction together with uncertainty.


In [ ]:
# Demonstrate final prediction output on the Random split test set.
# This uses the 90% split-conformal interval because it is the default practical setting.
random_result = catboost_results["Random"]
random_test_pred = random_result["test_pred"]
lower_90, upper_90 = interval_lookup[("Random", "Split Conformal", 0.90)]

random_test_outcomes = pd.DataFrame({
    "District": df.loc[X_te_r.index, "District"].values,
    "Season": df.loc[X_te_r.index, "Season"].values,
    "Crop Name": df.loc[X_te_r.index, "Crop Name"].values,
    "Actual AP Ratio": y_te_r.values,
    "Predicted AP Ratio": random_test_pred,
    "Lower 90": lower_90,
    "Upper 90": upper_90,
})
random_test_outcomes["Interval Width"] = random_test_outcomes["Upper 90"] - random_test_outcomes["Lower 90"]
random_test_outcomes["Covered"] = (
    (random_test_outcomes["Actual AP Ratio"] >= random_test_outcomes["Lower 90"]) &
    (random_test_outcomes["Actual AP Ratio"] <= random_test_outcomes["Upper 90"])
)
random_test_outcomes["Absolute Error"] = (
    random_test_outcomes["Actual AP Ratio"] - random_test_outcomes["Predicted AP Ratio"]
).abs()

summary_outcome = {
    "test_rows": len(random_test_outcomes),
    "coverage_90": random_test_outcomes["Covered"].mean(),
    "avg_interval_width": random_test_outcomes["Interval Width"].mean(),
    "avg_absolute_error": random_test_outcomes["Absolute Error"].mean(),
}
print("Random split test-set outcome summary")
print(summary_outcome)

random_test_outcomes.round({
    "Actual AP Ratio": 4,
    "Predicted AP Ratio": 4,
    "Lower 90": 4,
    "Upper 90": 4,
    "Interval Width": 4,
    "Absolute Error": 4,
}).head(20)


In [ ]:
# A compact visual demonstration for a subset of test samples.
# Sorted by absolute error so both easy and difficult cases are visible.
plot_demo = random_test_outcomes.sort_values("Absolute Error").copy()
if len(plot_demo) > 30:
    easy_cases = plot_demo.head(15)
    hard_cases = plot_demo.tail(15)
    plot_demo = pd.concat([easy_cases, hard_cases], axis=0)

plot_demo = plot_demo.reset_index(drop=True)
x = np.arange(len(plot_demo))
yerr = np.vstack([
    plot_demo["Predicted AP Ratio"] - plot_demo["Lower 90"],
    plot_demo["Upper 90"] - plot_demo["Predicted AP Ratio"],
])

fig, ax = plt.subplots(figsize=(14, 6))
ax.errorbar(
    x,
    plot_demo["Predicted AP Ratio"],
    yerr=yerr,
    fmt="o",
    color="steelblue",
    ecolor="lightsteelblue",
    elinewidth=2,
    capsize=3,
    label="Prediction + 90% interval",
)
ax.scatter(
    x,
    plot_demo["Actual AP Ratio"],
    color="crimson",
    s=28,
    label="Actual AP Ratio",
    zorder=3,
)
ax.set_xticks(x)
ax.set_xticklabels(plot_demo["Crop Name"], rotation=60, ha="right")
ax.set_ylabel("AP Ratio")
ax.set_title("Held-Out Test Examples: Predictions with 90% Conformal Intervals", fontweight="bold")
ax.grid(axis="y", alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Optional export for reporting or API-demo examples.
OUTCOME_PATH = "../dataset/random_test_conformal_outcomes.csv"
random_test_outcomes.to_csv(OUTCOME_PATH, index=False)
print(f"Saved test-set prediction outcomes -> {OUTCOME_PATH}")


## 8. Coverage by Subgroup

Global conformal prediction targets marginal coverage. That does not guarantee every crop or district has the same coverage. The next diagnostic checks subgroup coverage for the random split at the 90% level.


In [ ]:
def subgroup_coverage(y_true, lower, upper, groups, min_count=10):
    group_df = pd.DataFrame({
        "y": np.asarray(y_true, dtype=float),
        "lower": np.asarray(lower, dtype=float),
        "upper": np.asarray(upper, dtype=float),
        "group": pd.Series(groups).astype(str).values,
    })
    group_df["covered"] = (group_df["y"] >= group_df["lower"]) & (group_df["y"] <= group_df["upper"])
    group_df["width"] = group_df["upper"] - group_df["lower"]
    summary = (
        group_df.groupby("group")
                .agg(n=("covered", "size"), coverage=("covered", "mean"), avg_width=("width", "mean"))
                .reset_index()
    )
    return summary[summary["n"] >= min_count].sort_values("coverage")


lower_90, upper_90 = interval_lookup[("Random", "Split Conformal", 0.90)]

crop_coverage_90 = subgroup_coverage(
    y_te_r, lower_90, upper_90,
    df.loc[X_te_r.index, "Crop Name"],
    min_count=10,
)

district_coverage_90 = subgroup_coverage(
    y_te_r, lower_90, upper_90,
    df.loc[X_te_r.index, "District"],
    min_count=5,
)

print("Lowest crop-wise coverage groups, Random split, 90% conformal interval")
display(crop_coverage_90.head(12))

print("Lowest district-wise coverage groups, Random split, 90% conformal interval")
display(district_coverage_90.head(12))


## 9. Optional Group-Conditional Conformal Prediction

Group-conditional conformal prediction estimates a separate residual quantile per group. Here the group is `Crop Name`. Crops with too few calibration samples use the global quantile as a fallback.

This can improve subgroup coverage parity, but it may produce wider intervals for crops whose calibration residuals are larger.


In [ ]:
def group_conditional_conformal(cal_y, cal_pred, cal_groups, test_pred, test_groups, alpha=0.10, min_cal=15):
    cal_frame = pd.DataFrame({
        "residual": np.abs(np.asarray(cal_y) - np.asarray(cal_pred)),
        "group": pd.Series(cal_groups).astype(str).values,
    })

    global_q = conformal_quantile(cal_frame["residual"], alpha)
    group_q = {}
    for group, group_frame in cal_frame.groupby("group"):
        if len(group_frame) >= min_cal:
            group_q[group] = conformal_quantile(group_frame["residual"], alpha)

    q_values = np.array([
        group_q.get(str(group), global_q)
        for group in pd.Series(test_groups).astype(str).values
    ])
    test_pred = np.asarray(test_pred, dtype=float)
    return test_pred - q_values, test_pred + q_values, group_q, global_q


random_result = catboost_results["Random"]
lower_gc, upper_gc, crop_quantiles, global_q = group_conditional_conformal(
    y_cal_r,
    random_result["cal_pred"],
    df.loc[X_cal_r.index, "Crop Name"],
    random_result["test_pred"],
    df.loc[X_te_r.index, "Crop Name"],
    alpha=0.10,
    min_cal=15,
)

print(f"Global 90% conformal half-width: {global_q:.4f}")
print(f"Crop-specific quantiles learned for {len(crop_quantiles)} crops")
print("Random split group-conditional interval metrics:")
print(interval_metrics(y_te_r, lower_gc, upper_gc))

group_conditional_crop_coverage = subgroup_coverage(
    y_te_r,
    lower_gc,
    upper_gc,
    df.loc[X_te_r.index, "Crop Name"],
    min_count=10,
)
group_conditional_crop_coverage.head(12)


## 10. MAPIE Note

MAPIE can also implement model-agnostic conformal prediction. It is optional here because it is not currently listed in `requirements.txt` and may not be installed in every environment.

The manual split-conformal implementation above is intentionally explicit so the notebook remains reproducible without adding another dependency.


In [ ]:
if MAPIE_AVAILABLE:
    print("MAPIE is installed. You can add a MapieRegressor comparison if desired.")
else:
    print("MAPIE is not installed; using the manual split-conformal implementation above.")


## 11. Takeaways

- Split conformal intervals are calibrated using validation residuals, not test residuals.
- The Gaussian baseline can be narrower or wider depending on residual skew and outliers, but it does not provide distribution-free coverage.
- OOD coverage should be interpreted empirically because spatial and season splits intentionally violate the i.i.d. assumption.
- Group-conditional conformal intervals help diagnose whether some crops need wider intervals than the global conformal quantile provides.
